<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/gp_model_first_3_epoch/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers torch evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import warnings
import evaluate
import pandas as pd
import numpy as np
import os
import re
import pickle
import gc
import torch
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from tqdm import tqdm
from google.colab import drive
from transformers import EarlyStoppingCallback

drive.mount('/content/drive')

with open('/content/drive/MyDrive/datasets/europarl_normalized.pkl', 'rb') as f:
    df = pickle.load(f)

Mounted at /content/drive


In [ ]:
print(df['gender'].value_counts())

min_count = df['gender'].value_counts().min()

df = (
    df.groupby('gender', group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=63))
    .reset_index(drop=True)
)

print(df['gender'].value_counts())

gender
male      740894
female    360361
Name: count, dtype: int64
gender
female    360361
male      360361
Name: count, dtype: int64


<ipython-input-3-e34e100695cf>:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=63))


In [ ]:
# encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['gender'])

df

,text,gender,label
0,"Cooperation is not only in our interest, but i...",female,0
1,"During the committee's first meeting, we enter...",female,0
2,"Towards the end of the debate on the matter, a...",female,0
3,"After all, something is obviously being set in...",female,0
4,"Today, the new Member States in particular req...",female,0
...,...,...,...
720717,"Their function, as I understand it, is to diss...",male,1
720718,That is my first point.,male,1
720719,What we need to do tomorrow is to accept the c...,male,1
720720,in writing. I welcome this report which:,male,1


In [ ]:
print(label_encoder.classes_)
print(df['label'].value_counts())

['female' 'male']
label
0    360361
1    360361
Name: count, dtype: int64


In [ ]:
# early stopping by overriding eval_f1 and eval_loss metrics to prevent over fitting
class DualMetricEarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3, min_delta_f1=1e-7, min_delta_loss=1e-7, tolerance_f1=0.02, tolerance_loss=0.02):
        self.patience = patience
        self.counter = 0
        self.best_f1 = None
        self.best_loss = None
        self.min_delta_f1 = min_delta_f1
        self.min_delta_loss = min_delta_loss
        self.tolerance_f1 = tolerance_f1
        self.tolerance_loss = tolerance_loss

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_f1 = metrics.get("eval_f1")
        current_loss = metrics.get("eval_loss")

        if current_f1 is None or current_loss is None:
            return control

        if self.best_f1 is None or self.best_loss is None:
            self.best_f1 = current_f1
            self.best_loss = current_loss
            self.counter = 0
        else:
            f1_improved = current_f1 > self.best_f1 + self.min_delta_f1
            f1_decline_ok = current_f1 >= self.best_f1 - self.tolerance_f1

            loss_improved = current_loss < self.best_loss - self.min_delta_loss
            loss_decline_ok = current_loss <= self.best_loss + self.tolerance_loss

            if f1_improved or loss_improved or (f1_decline_ok and loss_decline_ok):
                self.best_f1 = max(self.best_f1, current_f1)
                self.best_loss = min(self.best_loss, current_loss)
                self.counter = 0
            else:
                self.counter += 1
                print(f"[EarlyStopping] No acceptable improvement. Patience {self.counter}/{self.patience}")

        if self.counter >= self.patience:
            print(f"[EarlyStopping] Triggered at epoch {state.epoch}.")
            control.should_training_stop = True

        return control

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    preds = logits.argmax(dim=1).to(device)
    labels = labels.to(device)

    num_classes = torch.max(labels).item() + 1
    f1_total = 0.0
    precision_total = 0.0
    recall_total = 0.0

    for cls in range(num_classes):
        tp = ((preds == cls) & (labels == cls)).sum()
        fp = ((preds == cls) & (labels != cls)).sum()
        fn = ((preds != cls) & (labels == cls)).sum()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        precision_total += precision
        recall_total += recall
        f1_total += f1

    macro_precision = precision_total / num_classes
    macro_recall = recall_total / num_classes
    macro_f1 = f1_total / num_classes

    accuracy = (preds == labels).sum().float() / labels.shape[0]

    return {
        "f1": macro_f1.item(),
        "precision": macro_precision.item(),
        "recall": macro_recall.item(),
        "accuracy": accuracy.item()
    }

# which model saved
class PrintBestModelCallback(TrainerCallback):
    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n[INFO] Best model was at step {state.best_step} with best {args.metric_for_best_model}: {state.best_metric}")

# split dataset
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.21, random_state=63)

# tokenization for huggingfaceapi
train_df = {"text": list(X_train), "label": list(y_train)}
test_df = {"text": list(X_test), "label": list(y_test)}
train_dataset = Dataset.from_dict(train_df)
eval_dataset = Dataset.from_dict(test_df)
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large", use_fast=False)
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

#load model
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-large", num_labels=2)

# define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/models/gp_checkpoints",
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    learning_rate=2.66e-6,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=250,
    save_total_limit=None,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=1,
    fp16=True,
)

# define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks = [DualMetricEarlyStoppingCallback(
    patience=14,
    min_delta_f1=1e-7,
    min_delta_loss=1e-7,
    tolerance_f1=0.05,
    tolerance_loss=0.03
    ),
                 PrintBestModelCallback()]
)

# training model
trainer.train(resume_from_checkpoint=True)

# saving model
save_path = "/content/drive/MyDrive/models/gp_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-6-298dab706e3d>:141: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
3500,0.673200,0.664638,0.588782,0.589307,0.589046,0.589110
3750,0.668300,0.665101,0.584472,0.590014,0.587387,0.587591
4000,0.668300,0.664145,0.591748,0.592417,0.592119,0.592050
4250,0.667300,0.665436,0.586473,0.591427,0.589050,0.589242
4500,0.666400,0.665274,0.590652,0.591195,0.590922,0.590987
4750,0.665200,0.662897,0.593864,0.593984,0.593935,0.593907
5000,0.665000,0.662555,0.593030,0.593271,0.593168,0.593127
5250,0.667000,0.663973,0.593614,0.593821,0.593733,0.593696
5500,0.663700,0.662558,0.592886,0.593852,0.593365,0.593451
5750,0.664100,0.661425,0.594208,0.594515,0.594354,0.594402


Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
3500,0.673200,0.664638,0.588782,0.589307,0.589046,0.589110
3750,0.668300,0.665101,0.584472,0.590014,0.587387,0.587591
4000,0.668300,0.664145,0.591748,0.592417,0.592119,0.592050
4250,0.667300,0.665436,0.586473,0.591427,0.589050,0.589242
4500,0.666400,0.665274,0.590652,0.591195,0.590922,0.590987
4750,0.665200,0.662897,0.593864,0.593984,0.593935,0.593907
5000,0.665000,0.662555,0.593030,0.593271,0.593168,0.593127
5250,0.667000,0.663973,0.593614,0.593821,0.593733,0.593696
5500,0.663700,0.662558,0.592886,0.593852,0.593365,0.593451
5750,0.664100,0.661425,0.594208,0.594515,0.594354,0.594402


In [ ]:
logs = trainer.state.log_history
log_df = pd.DataFrame(logs)
display(log_df)

In [ ]:
# visualization
predictions = trainer.predict(eval_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)

# classification report
print("Classification Report:")
print(classification_report(y_test, predicted_labels, target_names=label_encoder.classes_))

# confusion matrix
conf_matrix = confusion_matrix(y_test, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()